# Notebook 1: Run Database Migrations

This notebook executes the SQL scripts located in the `sql/` directory to create and alter the necessary tables in your database.

The migration process is **idempotent**, meaning it can be run multiple times without causing errors. It uses `CREATE TABLE IF NOT EXISTS` and `ALTER TABLE ... ADD COLUMN IF NOT EXISTS` to ensure the schema is up-to-date without breaking an existing database.

**Expected Outcome:**
- All tables defined in `sql/00_create_schema.sql` will be created.
- All columns defined in `sql/01_alter_columns.sql` will be added to their respective tables.

In [ ]:
import sys
import os

# Add project root to path to import our modules
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    
from src.db import run_migrations, get_engine
from sqlalchemy import text

### 1. Execute Migrations

In [ ]:
try:
    print("Starting database migrations...")
    run_migrations()
    print("\nMigrations script finished execution.")
except Exception as e:
    print(f"❌ An error occurred during migrations: {e}")

### 2. Verify Table Creation

Let's check if the main tables were created.

In [ ]:
expected_tables = [
    'congestion_data',
    'metar_raw',
    'passenger_forecast_raw',
    'passenger_forecast_t1',
    'congestion_features_t1',
    'policy_evaluation_log',
    'experiment_runs',
    'experiment_metrics'
]

try:
    engine = get_engine()
    with engine.connect() as connection:
        # The specific query to show tables can vary between DBs (e.g., SQLite vs MySQL)
        # This is for MySQL.
        result = connection.execute(text("SHOW TABLES;"))
        existing_tables = [row[0] for row in result.fetchall()]
    
    print("Checking for expected tables:")
    all_found = True
    for table in expected_tables:
        if table in existing_tables:
            print(f"  - [✓] Found table: {table}")
        else:
            print(f"  - [✗] Missing table: {table}")
            all_found = False
    
    if all_found:
        print("\n✅ All expected tables exist in the database.")
    else:
        print("\n❌ Some tables are missing. Please check the migration logs above.")
        
except Exception as e:
    print(f"❌ Could not verify tables due to an error: {e}")